In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# MedAssist AI: Context Optimization for Clinical Decision Support — Implementation Notebook

# MedAssist AI Case Study
## Optimizing Context Quality for Clinical Decision Support

In this notebook, you will build a complete context optimization and evaluation pipeline for a clinical decision support system. You will implement compression, dynamic budgeting, RAGAS evaluation, A/B testing, and production monitoring.

**Scenario:** MedAssist AI's answer quality has degraded over 8 months. Faithfulness dropped from 0.94 to 0.82, context precision from 0.87 to 0.69, and latency doubled.

## 3.1 Environment Setup

In [ ]:
!pip install -q numpy scikit-learn matplotlib pandas tiktoken scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tiktoken
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy import stats
from datetime import datetime, timedelta
from collections import Counter

enc = tiktoken.encoding_for_model("gpt-4")
np.random.seed(42)
print("Setup complete.")

## 3.2 Simulated Clinical Data

In [ ]:
# Simulated medical documents
medical_docs = [
    "Amoxicillin is a penicillin-type antibiotic used to treat bacterial infections. Standard adult dose is 500mg every 8 hours. Contraindicated in patients with penicillin allergy.",
    "Metformin is first-line therapy for type 2 diabetes. Starting dose 500mg twice daily. Contraindicated in severe renal impairment (eGFR < 30). Monitor renal function annually.",
    "Chest pain differential includes acute coronary syndrome, pulmonary embolism, pneumothorax, and musculoskeletal causes. ECG and troponin are first-line investigations.",
    "Warfarin requires regular INR monitoring. Target INR 2.0-3.0 for atrial fibrillation. Many drug interactions including NSAIDs, antibiotics, and herbal supplements.",
    "Sepsis management follows the Surviving Sepsis Campaign guidelines. Hour-1 bundle: lactate, blood cultures, broad-spectrum antibiotics, IV fluids 30ml/kg, vasopressors if MAP < 65.",
    "Asthma exacerbation management: inhaled short-acting beta-agonists first line, add ipratropium if severe, systemic corticosteroids within 1 hour, consider magnesium sulfate IV.",
    "Deep vein thrombosis diagnosis uses Wells score for pre-test probability. D-dimer for low probability. Compression ultrasound for moderate-high probability.",
    "Acute kidney injury staging uses KDIGO criteria. Stage 1: creatinine 1.5-1.9x baseline. Stage 2: 2.0-2.9x. Stage 3: 3.0x or creatinine > 4.0.",
    "Hypertensive emergency defined as BP > 180/120 with end-organ damage. Treat with IV labetalol or nicardipine. Target 25% reduction in first hour.",
    "The seasonal flu vaccine composition is updated annually based on WHO recommendations. It typically covers two influenza A and two influenza B strains."
]

# Simulated test queries
test_queries = [
    {"query": "What is the first-line treatment for type 2 diabetes?", "relevant_docs": [1], "domain": "pharmacology"},
    {"query": "How do you manage sepsis in the emergency department?", "relevant_docs": [4], "domain": "emergency"},
    {"query": "What are the contraindications for warfarin?", "relevant_docs": [3], "domain": "pharmacology"},
    {"query": "How do you diagnose a pulmonary embolism?", "relevant_docs": [2, 6], "domain": "diagnostics"},
    {"query": "What is the KDIGO staging for acute kidney injury?", "relevant_docs": [7], "domain": "diagnostics"},
]

print(f"Loaded {len(medical_docs)} documents, {len(test_queries)} test queries")

## 3.3 Baseline Measurement

In [ ]:
def simple_retrieve(query, docs, top_k=5):
    """Retrieve top-k documents by TF-IDF similarity."""
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf = vectorizer.fit_transform([query] + docs)
    sims = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
    top_idx = np.argsort(sims)[-top_k:][::-1]
    return [(idx, docs[idx], sims[idx]) for idx in top_idx]

def compute_precision(retrieved_indices, relevant_indices):
    labels = [i in relevant_indices for i in retrieved_indices]
    if not any(labels): return 0.0
    precs = []
    count = 0
    for k, rel in enumerate(labels, 1):
        if rel:
            count += 1
            precs.append(count / k)
    return sum(precs) / len(precs) if precs else 0.0

# Baseline evaluation
print("=== BASELINE EVALUATION ===\n")
baseline_results = []
for tq in test_queries:
    retrieved = simple_retrieve(tq["query"], medical_docs, top_k=5)
    r_indices = [r[0] for r in retrieved]
    precision = compute_precision(r_indices, tq["relevant_docs"])
    tokens = sum(len(enc.encode(r[1])) for r in retrieved)
    baseline_results.append({"query": tq["query"][:50], "precision": precision, "tokens": tokens})
    print(f"  Q: {tq['query'][:50]}...")
    print(f"    Precision: {precision:.3f}, Tokens: {tokens}")

avg_prec = np.mean([r["precision"] for r in baseline_results])
avg_tok = np.mean([r["tokens"] for r in baseline_results])
print(f"\nBaseline: Avg Precision={avg_prec:.3f}, Avg Tokens={avg_tok:.0f}")

## 3.4 TODO: Build the Compression Pipeline

In [ ]:
def medical_extractive_compress(document, query, keep_ratio=0.5):
    """
    TODO: Implement extractive compression for medical documents.

    Requirements:
    1. Split document into sentences (handle medical abbreviations like "mg", "ml/kg")
    2. Score each sentence by TF-IDF similarity to query
    3. Keep top keep_ratio fraction of sentences
    4. Preserve original sentence order
    5. Never remove sentences containing drug dosages or contraindications

    Hint: Check if sentence contains keywords like "dose", "mg", "contraindicated"
    before allowing it to be removed.

    Returns: compressed document string
    """
    # YOUR CODE HERE
    # sentences = document.split('. ')
    # critical_keywords = ["dose", "mg", "contraindicated", "warning", "alert"]
    # ...
    return document  # Replace with compressed version

# Test
compressed = medical_extractive_compress(medical_docs[0], "What is amoxicillin dosing?", keep_ratio=0.5)
print(f"Original: {len(enc.encode(medical_docs[0]))} tokens")
print(f"Compressed: {len(enc.encode(compressed))} tokens")

## 3.5 TODO: Dynamic Budget Allocator

In [ ]:
class ClinicalBudgetAllocator:
    """
    TODO: Implement dynamic token budget allocation for clinical queries.

    Query complexity levels:
    - 'triage': Simple factual queries (e.g., "What is the dose of X?")
      -> Budget: guidelines=3K, evidence=15K, history=2K
    - 'differential': Diagnostic reasoning queries
      -> Budget: guidelines=3K, evidence=30K, history=10K
    - 'treatment_plan': Complex treatment queries
      -> Budget: guidelines=3K, evidence=40K, history=15K

    Methods to implement:
    - classify_query(query) -> str: classify into triage/differential/treatment_plan
    - allocate(query, window_size=128000, reserved=35000) -> dict
    """
    def classify_query(self, query):
        # YOUR CODE HERE
        return "triage"

    def allocate(self, query, window_size=128000, reserved=35000):
        # YOUR CODE HERE
        return {"guidelines": 3000, "evidence": 15000, "history": 2000}

# Test
allocator = ClinicalBudgetAllocator()
for q in test_queries:
    complexity = allocator.classify_query(q["query"])
    budget = allocator.allocate(q["query"])
    print(f"  [{complexity}] {q['query'][:50]}... -> {budget}")

## 3.6 TODO: RAGAS Evaluation

In [ ]:
class ClinicalRAGASEvaluator:
    """
    TODO: Implement RAGAS evaluation with clinical extensions.

    Standard metrics:
    - context_precision: average precision over ranked chunks
    - faithfulness: fraction of answer claims supported by context

    Clinical extensions:
    - drug_safety: Are all mentioned drugs present in the context?
    - contraindication_coverage: Are relevant contraindications mentioned?

    Methods to implement:
    - evaluate(question, context_chunks, relevance_labels,
               answer, answer_claims) -> dict
    """
    def evaluate(self, question, chunks, labels, answer, claims):
        # YOUR CODE HERE
        return {"precision": 0.0, "faithfulness": 0.0, "relevancy": 0.0}

# Test
evaluator = ClinicalRAGASEvaluator()
result = evaluator.evaluate(
    "What is the first-line treatment for type 2 diabetes?",
    [medical_docs[1], medical_docs[9]],
    [True, False],
    "Metformin is first-line for type 2 diabetes, starting at 500mg twice daily.",
    ["Metformin is first-line", "Starting dose 500mg twice daily"]
)
print(f"Evaluation: {result}")

## 3.7 TODO: A/B Test the Optimized Pipeline

In [ ]:
def clinical_ab_test(baseline_results, optimized_results, metric="precision"):
    """
    TODO: Run a two-proportion z-test comparing baseline vs optimized.

    Steps:
    1. Binarize metric scores (>= 0.8 = success, < 0.8 = failure)
    2. Count successes in each group
    3. Run two_proportion_ztest
    4. Return result dict with significance, lift, CI

    Args:
        baseline_results: list of metric dicts from baseline
        optimized_results: list of metric dicts from optimized
        metric: which metric to compare

    Returns: dict with z_score, p_value, significant, lift
    """
    # YOUR CODE HERE
    return {"z_score": 0, "p_value": 1, "significant": False, "lift": 0}

# Simulate optimized results (improvement)
optimized_results = []
for r in baseline_results:
    optimized_results.append({**r, "precision": min(1.0, r["precision"] + np.random.uniform(0.1, 0.2))})

# test_result = clinical_ab_test(baseline_results, optimized_results)
# print(f"A/B Test: {test_result}")

## 3.8 TODO: Production Monitor

In [ ]:
class ClinicalMonitor:
    """
    TODO: Implement production monitoring with clinical alert thresholds.

    Metrics to track with EMA:
    - faithfulness: alpha=0.15, alert below 0.90 (CRITICAL for clinical safety)
    - precision: alpha=0.2, alert below 0.80
    - token_usage: alpha=0.3, alert above 75000
    - latency_ms: alpha=0.3, alert above 2500

    Compound alerts:
    - If faithfulness AND precision both below alert thresholds: EMERGENCY
    - If any metric in alert state for 3+ consecutive days: ESCALATE

    Methods to implement:
    - record(timestamp, metrics) -> list of alerts
    - status() -> dict of metric statuses
    - report() -> formatted string report
    """
    def __init__(self):
        # YOUR CODE HERE
        self.ema_vals = {}
        self.alerts = []

    def record(self, ts, metrics):
        # YOUR CODE HERE
        return []

    def status(self):
        # YOUR CODE HERE
        return {}

# Test
# monitor = ClinicalMonitor()
# monitor.record(datetime.now(), {"faithfulness": 0.91, "precision": 0.85})
# print(monitor.status())

## 3.9 Results Analysis

In [ ]:
# Summary comparison
print("=" * 60)
print("  MEDASSIST AI: OPTIMIZATION RESULTS")
print("=" * 60)

metrics_table = [
    ("Faithfulness", 0.82, 0.93, "> 0.92"),
    ("Context Precision", 0.69, 0.87, "> 0.85"),
    ("Token Usage", "85K", "52K", "< 70K"),
    ("Latency (p99)", "4.2s", "2.1s", "< 2.5s"),
    ("Cost/query", "$0.062", "$0.038", "< $0.05"),
]

print(f"\n  {'Metric':<22} {'Before':>10} {'After':>10} {'Target':>10} {'Status':>10}")
print("  " + "-" * 62)
for name, before, after, target in metrics_table:
    status = "PASS" if True else "FAIL"
    print(f"  {name:<22} {str(before):>10} {str(after):>10} {target:>10} {status:>10}")

print(f"\n  Annual liability exposure: $8.2M -> $1.4M (82.9% reduction)")
print("=" * 60)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Before/After bar chart
ax1 = axes[0]
metrics = ['Faithfulness', 'Precision', 'Relevancy']
before = [0.82, 0.69, 0.85]
after = [0.93, 0.87, 0.90]
x = np.arange(len(metrics))
ax1.bar(x - 0.2, before, 0.35, label='Before', color='#EA4335', alpha=0.8)
ax1.bar(x + 0.2, after, 0.35, label='After', color='#34A853', alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)
ax1.set_ylim(0, 1.1)
ax1.legend()
ax1.set_title('Quality Metrics')

# Token usage
ax2 = axes[1]
ax2.bar(['Before', 'After'], [85000, 52000], color=['#EA4335', '#34A853'], alpha=0.8)
ax2.axhline(y=70000, color='orange', linestyle='--', label='Target')
ax2.set_ylabel('Tokens')
ax2.set_title('Token Usage')
ax2.legend()

# Cost
ax3 = axes[2]
ax3.bar(['Before', 'After'], [0.062, 0.038], color=['#EA4335', '#34A853'], alpha=0.8)
ax3.axhline(y=0.05, color='orange', linestyle='--', label='Budget')
ax3.set_ylabel('USD per query')
ax3.set_title('Cost per Query')
ax3.legend()

plt.suptitle('MedAssist AI: Before vs After Optimization', fontsize=14)
plt.tight_layout()
plt.savefig('medassist_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Congratulations!

You have completed the MedAssist AI case study. The key insight: **systematic optimization with rigorous evaluation** transformed a degrading clinical AI system into one that exceeds quality targets while reducing costs by nearly 40%.